# Final Project: Interactive Figure Notebook
Welcome to the interactive exploration notebook! This notebook allows you to interactively explore our custom implementations of machine learning models and unsupervised analysis tools.

Using the sliders below, you can visualize the behavior of:
1. **AdaBoost Scaling:** Observe how training and testing error scale with the number of estimators.
2. **Random Forest Scaling:** Tune forest depth and size to see accuracy and OOB score live.
3. **Bias-Variance Decomposition:** Visualize how model family choice affects the bias-variance profile.
4. **Unsupervised Clustering:** Tune K-Means and DBSCAN live, and see clusters vs. true labels on PCA 2D projections.
5. **Noise Robustness:** Toggle label corruption to see how each model degrades under noise.

---
## Import required packages


In [ ]:
%matplotlib inline
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual

# Locate project root dynamically
project_root = os.getcwd()
while project_root != "/" and not os.path.exists(os.path.join(project_root, "src")):
    project_root = os.path.abspath(os.path.join(project_root, ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

# Custom imports
from src.utils.preprocessing import load_breast_cancer, load_adult, load_covertype, load_mnist_subset, train_test_split, standardize
from src.boosting.adaboost import AdaBoostClassifier
from src.bagging.random_forest import RandomForestClassifier
from src.trees.decision_tree import DecisionTree
from src.unsupervised.pca import PCA
from src.unsupervised.kmeans import KMeans
from src.unsupervised.dbscan import DBSCAN
from src.metrics.evaluation import accuracy_score, precision_recall_f1
from sklearn.metrics import adjusted_rand_score

print("Imports successful! Custom implementations loaded.")


---
## Section 1: AdaBoost Scaling Explorer
In this section, you can explore how AdaBoost training and testing errors change as the number of boosting estimators increases. 
We use our custom `AdaBoostClassifier` with decision stumps on the selected dataset.


In [ ]:
# Pre-load and cache all 4 datasets for fast instant interactivity
datasets_cache = {}

def get_dataset_splits(name):
    if name == "Breast Cancer":
        X, y = load_breast_cancer()
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    elif name == "Adult Income":
        X_tr_f, X_te_f, y_tr_f, y_te_f = load_adult()
        X_all = np.vstack([X_tr_f, X_te_f])
        y_all = np.concatenate([y_tr_f, y_te_f])
        rng = np.random.default_rng(42)
        idx = rng.choice(len(X_all), size=min(1000, len(X_all)), replace=False)
        X_tr, X_te, y_tr, y_te = train_test_split(X_all[idx], y_all[idx], test_size=0.2, random_state=42)
    elif name == "MNIST (3 vs 8)":
        X, y = load_mnist_subset(n_samples=1000, classes=("3", "8"), random_state=42)
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    elif name == "Covertype":
        X, y = load_covertype(n_samples=2000, random_state=42)
        # Covertype is multi-class; filter to binary for binary classifiers (0 vs 1)
        mask = (y == 0) | (y == 1)
        X_bin = X[mask]
        y_bin = y[mask]
        rng = np.random.default_rng(42)
        idx = rng.choice(len(X_bin), size=min(1000, len(X_bin)), replace=False)
        X_tr, X_te, y_tr, y_te = train_test_split(X_bin[idx], y_bin[idx], test_size=0.2, random_state=42)
        
    X_tr_s, X_te_s = standardize(X_tr, X_te)
    return X_tr_s, X_te_s, y_tr, y_te

print("Pre-loading datasets (this may take up to 10 seconds)...")
for name in ["Breast Cancer", "Adult Income", "MNIST (3 vs 8)", "Covertype"]:
    datasets_cache[name] = get_dataset_splits(name)
print("All 4 datasets loaded and cached successfully!")

# Unpack Breast Cancer for Section 3 & 4 compatibility
X_bc_tr, X_bc_te, y_bc_tr, y_bc_te = datasets_cache["Breast Cancer"]

def plot_adaboost_scaling(dataset, n_estimators):
    X_tr, X_te, y_tr, y_te = datasets_cache[dataset]

    # Fit a single model up to max estimators to make interactive sliding faster
    model = AdaBoostClassifier(n_estimators=n_estimators, random_state=42)
    model.fit(X_tr, y_tr)
    
    # Evaluate staged errors
    train_errors = []
    test_errors = []
    
    for y_pred_tr in model.staged_predict(X_tr):
        train_errors.append(1.0 - accuracy_score(y_tr, y_pred_tr))
    for y_pred_te in model.staged_predict(X_te):
        test_errors.append(1.0 - accuracy_score(y_te, y_pred_te))
        
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(train_errors) + 1), train_errors, label="Train Error", color="blue", lw=2)
    plt.plot(range(1, len(test_errors) + 1), test_errors, label="Test Error", color="red", lw=2)
    plt.axvline(x=n_estimators, color="gray", linestyle="--", label=f"Current Estimators: {n_estimators}")
    
    plt.xlabel("Number of Estimators")
    plt.ylabel("Error Rate")
    plt.title(f"AdaBoost Error Curves on {dataset}")
    plt.legend()
    plt.grid(True)
    plt.show()

widgets.interact(
    plot_adaboost_scaling,
    dataset=widgets.Dropdown(options=["Breast Cancer", "Adult Income", "MNIST (3 vs 8)", "Covertype"], value="Breast Cancer", description="Dataset:"),
    n_estimators=widgets.IntSlider(min=5, max=100, step=5, value=50, description="Estimators:")
);


---
## Section 2: Random Forest Depth vs. Accuracy
Vary the tree depth and total estimators in our custom `RandomForestClassifier` to see how it affects classification accuracy and Out-of-Bag (OOB) generalization.


In [ ]:
def plot_rf_scaling(dataset, n_estimators, max_depth, oob_score):
    X_tr, X_te, y_tr, y_te = datasets_cache[dataset]
        
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth if max_depth > 0 else None,
        oob_score=oob_score,
        random_state=42
    )
    rf.fit(X_tr, y_tr)
    
    train_acc = accuracy_score(y_tr, rf.predict(X_tr))
    test_acc = accuracy_score(y_te, rf.predict(X_te))
    
    metrics = ["Train Accuracy", "Test Accuracy"]
    values = [train_acc, test_acc]
    colors = ["blue", "green"]
    
    if oob_score and rf.oob_score_ is not None:
        metrics.append("OOB Accuracy")
        values.append(rf.oob_score_)
        colors.append("orange")
        
    plt.figure(figsize=(8, 4))
    bars = plt.bar(metrics, values, color=colors, width=0.5)
    plt.ylim(0, 1.05)
    plt.ylabel("Accuracy")
    plt.title(f"Random Forest Performance ({n_estimators} estimators, depth={max_depth if max_depth > 0 else 'None'})")
    
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, height + 0.01, f"{height:.4f}", ha="center", va="bottom")
        
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

widgets.interact(
    plot_rf_scaling,
    dataset=widgets.Dropdown(options=["Breast Cancer", "Adult Income", "MNIST (3 vs 8)", "Covertype"], value="Breast Cancer", description="Dataset:"),
    n_estimators=widgets.IntSlider(min=5, max=50, step=5, value=25, description="Estimators:"),
    max_depth=widgets.IntSlider(min=0, max=15, step=1, value=5, description="Max Depth (0=None):"),
    oob_score=widgets.Checkbox(value=True, description="Enable OOB Score")
);


---
## Section 3: Bias–Variance Decomposition
Observe how model complexity impacts the Bias² and Variance components of total error. 
We draw $B$ bootstrap replicates of the training set, train the selected model type, and evaluate predictions on a fixed test set.


In [ ]:
from experiments.bias_variance import bias_variance_decomposition

# Pre-define helper functions for constructors to avoid lambda lints
def get_dt(depth):
    return lambda: DecisionTree(max_depth=depth, random_state=42)

def get_ada(n_est):
    return lambda: AdaBoostClassifier(n_estimators=n_est, random_state=42)

def get_rf(n_est, depth):
    return lambda: RandomForestClassifier(n_estimators=n_est, max_depth=depth, random_state=42)

def plot_bias_variance(model_type, bootstraps):
    # Set up model constructor based on parameters
    if model_type == "Decision Tree (stump, depth=1)":
        model_fn = get_dt(1)
    elif model_type == "Decision Tree (deep, depth=10)":
        model_fn = get_dt(10)
    elif model_type == "AdaBoost (10 estimators)":
        model_fn = get_ada(10)
    elif model_type == "AdaBoost (50 estimators)":
        model_fn = get_ada(50)
    elif model_type == "Random Forest (10 trees, depth=5)":
        model_fn = get_rf(10, 5)
    elif model_type == "Random Forest (50 trees, depth=10)":
        model_fn = get_rf(50, 10)
        
    print("Computing Bias-Variance decomposition (please wait, running bootstrap)...")
    bias_sq, variance = bias_variance_decomposition(
        model_fn,
        X_bc_tr,
        y_bc_tr,
        X_bc_te,
        y_bc_te,
        n_bootstraps=bootstraps,
        random_state=42
    )
    
    total_error = bias_sq + variance
    
    plt.figure(figsize=(8, 4))
    categories = ["Bias²", "Variance", "Total Expected Error"]
    vals = [bias_sq, variance, total_error]
    colors = ["salmon", "lightblue", "purple"]
    
    bars = plt.bar(categories, vals, color=colors, width=0.5)
    plt.ylim(0, max(0.15, total_error * 1.2))
    plt.ylabel("Decomposition Value")
    plt.title(f"Bias-Variance Profile (bootstraps={bootstraps})")
    
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, height + 0.002, f"{height:.4f}", ha="center", va="bottom")
        
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

widgets.interact(
    plot_bias_variance,
    model_type=widgets.Dropdown(
        options=[
            "Decision Tree (stump, depth=1)",
            "Decision Tree (deep, depth=10)",
            "AdaBoost (10 estimators)",
            "AdaBoost (50 estimators)",
            "Random Forest (10 trees, depth=5)",
            "Random Forest (50 trees, depth=10)"
        ],
        value="Decision Tree (stump, depth=1)",
        description="Model Type:"
    ),
    bootstraps=widgets.IntSlider(min=10, max=50, step=10, value=30, description="Bootstraps:")
);


---
## Section 4: Unsupervised Clusters (K-Means & DBSCAN)
Here we project our features onto PCA 2D space and compare the True class labels against clustering results.
You can tune K-Means `k` and DBSCAN `eps` / `min_samples` live, and observe how cluster borders change and how the Adjusted Rand Index (ARI) responds.


In [ ]:
# Run PCA for visualization once
pca = PCA(n_components=2)
X_bc_2d = pca.fit_transform(X_bc_tr)

def plot_unsupervised(k, eps, min_samples):
    # Fit K-Means
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_bc_tr)
    ari_km = adjusted_rand_score(y_bc_tr, km.labels_)
    
    # Fit DBSCAN
    db = DBSCAN(eps=eps, min_samples=min_samples)
    db.fit(X_bc_tr)
    ari_db = adjusted_rand_score(y_bc_tr, db.labels_)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Panel 1: True Labels
    axes[0].scatter(X_bc_2d[:, 0], X_bc_2d[:, 1], c=y_bc_tr, cmap="viridis", alpha=0.8, edgecolor="k")
    axes[0].set_title("True Labels")
    
    # Panel 2: K-Means
    axes[1].scatter(X_bc_2d[:, 0], X_bc_2d[:, 1], c=km.labels_, cmap="tab10", alpha=0.8, edgecolor="k")
    axes[1].set_title(f"K-Means (k={k})\nARI: {ari_km:.4f}")
    
    # Panel 3: DBSCAN
    axes[2].scatter(X_bc_2d[:, 0], X_bc_2d[:, 1], c=db.labels_, cmap="tab10", alpha=0.8, edgecolor="k")
    axes[2].set_title(f"DBSCAN (eps={eps}, min={min_samples})\nARI: {ari_db:.4f}")
    
    for ax in axes:
        ax.set_xlabel("PC 1")
        ax.set_ylabel("PC 2")
        ax.grid(True, linestyle="--", alpha=0.5)
        
    plt.tight_layout()
    plt.show()

widgets.interact(
    plot_unsupervised,
    k=widgets.IntSlider(min=2, max=6, step=1, value=2, description="K-Means k:"),
    eps=widgets.FloatSlider(min=1.5, max=5.0, step=0.5, value=3.5, description="DBSCAN eps:"),
    min_samples=widgets.IntSlider(min=2, max=10, step=1, value=5, description="DBSCAN min:")
);


---
## Section 5: Noise Robustness Live View
Observe the robustness of ensemble algorithms under label corruption. 
We randomly flip a fraction $\eta$ of training labels, train the models, and test them on clean data.


In [ ]:
def plot_noise_robustness(dataset, noise_rate):
    X_tr, X_te, y_tr, y_te = datasets_cache[dataset]
        
    # Corrupt training labels
    rng = np.random.default_rng(42)
    n_samples = len(y_tr)
    n_flip = int(noise_rate * n_samples)
    
    y_corrupt = y_tr.copy()
    if n_flip > 0:
        flip_indices = rng.choice(n_samples, size=n_flip, replace=False)
        y_corrupt[flip_indices] = 1 - y_corrupt[flip_indices]
        
    # Train models
    ada = AdaBoostClassifier(n_estimators=50, random_state=42)
    ada.fit(X_tr, y_corrupt)
    ada_acc = accuracy_score(y_te, ada.predict(X_te))
    
    rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
    rf.fit(X_tr, y_corrupt)
    rf_acc = accuracy_score(y_te, rf.predict(X_te))
    
    # Plot accuracy horizontal comparison
    plt.figure(figsize=(8, 3))
    models = ["AdaBoost Accuracy", "Random Forest Accuracy"]
    accuracies = [ada_acc, rf_acc]
    colors = ["red", "green"]
    
    bars = plt.barh(models, accuracies, color=colors, height=0.4)
    plt.xlim(0, 1.05)
    plt.xlabel("Accuracy on Clean Test Set")
    plt.title(f"Model Performance under {noise_rate*100:.0f}% Label Noise")
    
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height()/2.0, f"{width:.4f}", ha="left", va="center")
        
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.show()

widgets.interact(
    plot_noise_robustness,
    dataset=widgets.Dropdown(options=["Breast Cancer", "Adult Income", "MNIST (3 vs 8)", "Covertype"], value="Breast Cancer", description="Dataset:"),
    noise_rate=widgets.FloatSlider(min=0.0, max=0.5, step=0.05, value=0.10, description="Noise Rate η:")
);
